In [ ]:
import kagglehub
import os
import shutil

# Step 1: Download dataset (goes to default cache directory)
download_path = kagglehub.dataset_download("tevecsystems/retail-sales-forecasting")
print("Downloaded to:", download_path)

# Step 2: Define your custom destination
destination = "content/retail-sales-forecasting"
os.makedirs(destination, exist_ok=True)

# Step 3: Move all files from the downloaded folder to your custom location
for filename in os.listdir(download_path):
    src = os.path.join(download_path, filename)
    dst = os.path.join(destination, filename)
    shutil.move(src, dst)

print(f"Dataset moved to: {destination}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import time

In [ ]:
# Load the CSV file
file_path = "/content/content/retail-sales-forecasting/mock_kaggle.csv"
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


In [ ]:
# Convert 'data' column to datetime
df['data'] = pd.to_datetime(df['data'])

# Set the 'data' column as the index (for time series operations)
df.set_index('data', inplace=True)

# Basic preprocessing check
print(df.info())
print(df.describe())

In [ ]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

In [ ]:
# Set style
sns.set(style="whitegrid")
plt.figure(figsize=(16, 16))

# 1. Line plot: Sales over time
plt.subplot(3, 2, 1)
df['venda'].plot(title='Daily Sales (venda)', color='steelblue')
plt.xlabel("Date")
plt.ylabel("Units Sold")

# 2. Line plot: Inventory over time
plt.subplot(3, 2, 2)
df['estoque'].plot(title='Inventory Level (estoque)', color='darkorange')
plt.xlabel("Date")
plt.ylabel("Stock Units")

# 3. Histogram: Sales distribution
plt.subplot(3, 2, 3)
sns.histplot(df['venda'], kde=True, color='skyblue')
plt.title("Sales Distribution")
plt.xlabel("Units Sold")
plt.ylabel("Frequency")

# 4. Scatter plot: Inventory vs Sales
plt.subplot(3, 2, 4)
sns.scatterplot(x='estoque', y='venda', data=df, color='teal')
plt.title("Inventory vs Sales")
plt.xlabel("Estoque (Stock)")
plt.ylabel("Venda (Sales)")

# 5. Correlation heatmap
plt.subplot(3, 2, 5)
sns.heatmap(df[['venda', 'estoque', 'preco']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")

# 6. Rolling average of sales (7-day)
plt.subplot(3, 2, 6)
df['venda'].rolling(window=7).mean().plot(title='7-Day Rolling Avg of Sales', color='green')
plt.xlabel("Date")
plt.ylabel("Rolling Sales")

plt.tight_layout()
plt.show()


# Feature Engineering for ML models

In [ ]:
# Time-based features
df['dayofweek'] = df.index.dayofweek
df['weekofyear'] = df.index.isocalendar().week
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['day'] = df.index.day
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# Lag features
df['venda_lag1'] = df['venda'].shift(1)
df['venda_lag7'] = df['venda'].shift(7)

# Rolling features
df['venda_roll7_mean'] = df['venda'].rolling(window=7).mean()
df['venda_roll7_std'] = df['venda'].rolling(window=7).std()

# Drop NA caused by lag/rolling
df_xgb = df.dropna().copy()


In [ ]:
# Save to CSV
df_xgb.to_csv("xgb_processed_venda_data.csv", index=True)

# Feature Engineering for LSTM

In [ ]:
def create_lstm_dataset(series, look_back=7):
    X, y = [], []
    for i in range(len(series) - look_back):
        X.append(series[i:i+look_back])
        y.append(series[i+look_back])
    return np.array(X), np.array(y)

# Normalize (LSTM works better with scaled data)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
venda_scaled = scaler.fit_transform(df[['venda']])

# Create supervised sequences
look_back = 7
X_lstm, y_lstm = create_lstm_dataset(venda_scaled, look_back=look_back)

print("LSTM Input shape:", X_lstm.shape)


In [ ]:
features = ['estoque', 'preco', 'dayofweek', 'weekofyear', 'month', 'quarter', 'day', 'is_weekend',
            'venda_lag1', 'venda_lag7', 'venda_roll7_mean', 'venda_roll7_std']

X_xgb = df_xgb[features]
y_xgb = df_xgb['venda']

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X_xgb, y_xgb, test_size=0.2, shuffle=False)


# Linear Regession

In [ ]:

start = time.time()
lr = LinearRegression()
lr.fit(X_train_xgb, y_train_xgb)
train_time_lr = time.time() - start

start = time.time()
y_pred_lr = lr.predict(X_test_xgb)
pred_time_lr = time.time() - start

r2_lr = r2_score(y_test_xgb, y_pred_lr)
mse_lr = mean_squared_error(y_test_xgb, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test_xgb, y_pred_lr)

print("📊 Linear Regression:")
print(f"R2: {r2_lr:.4f} | MSE: {mse_lr:.2f} | RMSE: {rmse_lr:.2f} | MAE: {mae_lr:.2f}")
print(f"⚡ Training Time: {train_time_lr:.4f} sec | Prediction Time: {pred_time_lr:.4f} sec\n")

# Random Forest

In [ ]:
start = time.time()
model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
model_rf.fit(X_train_xgb, y_train_xgb)
train_time_rf = time.time() - start

start = time.time()
y_pred_rf = model_rf.predict(X_test_xgb)
pred_time_rf = time.time() - start

r2_rf = r2_score(y_test_xgb, y_pred_rf)
mse_rf = mean_squared_error(y_test_xgb, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test_xgb, y_pred_rf)

print("📊 Random Forest:")
print(f"R2: {r2_rf:.4f} | MSE: {mse_rf:.2f} | RMSE: {rmse_rf:.2f} | MAE: {mae_rf:.2f}")
print(f"⚡ Training Time: {train_time_rf:.4f} sec | Prediction Time: {pred_time_rf:.4f} sec\n")

# XGBoost

In [ ]:
start = time.time()
model_xgb = xgb.XGBRegressor(random_state=42)
model_xgb.fit(X_train_xgb, y_train_xgb)
train_time_xgb = time.time() - start

start = time.time()
y_pred_xgb = model_xgb.predict(X_test_xgb)
pred_time_xgb = time.time() - start

r2_xgb = r2_score(y_test_xgb, y_pred_xgb)
mse_xgb = mean_squared_error(y_test_xgb, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
mae_xgb = mean_absolute_error(y_test_xgb, y_pred_xgb)

print("📊 XGBoost:")
print(f"R2: {r2_xgb:.4f} | MSE: {mse_xgb:.2f} | RMSE: {rmse_xgb:.2f} | MAE: {mae_xgb:.2f}")
print(f"⚡ Training Time: {train_time_xgb:.4f} sec | Prediction Time: {pred_time_xgb:.4f} sec\n")


# LSTM

In [ ]:
look_back = 7
X_lstm, y_lstm = create_lstm_dataset(venda_scaled, look_back=look_back)
split = int(len(X_lstm) * 0.8)
X_train_lstm, X_test_lstm = X_lstm[:split], X_lstm[split:]
y_train_lstm, y_test_lstm = y_lstm[:split], y_lstm[split:]

model_lstm = Sequential()
model_lstm.add(LSTM(50, input_shape=(look_back, 1)))
model_lstm.add(Dense(1))
model_lstm.compile(loss='mse', optimizer='adam')

start = time.time()
history = model_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)
train_time_lstm = time.time() - start

start = time.time()
y_pred_lstm = model_lstm.predict(X_test_lstm)
pred_time_lstm = time.time() - start

mse_lstm = mean_squared_error(y_test_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(y_test_lstm, y_pred_lstm)
r2_lstm = r2_score(y_test_lstm, y_pred_lstm)

print("📊 LSTM:")
print(f"R2: {r2_lstm:.4f} | MSE: {mse_lstm:.2f} | RMSE: {rmse_lstm:.2f} | MAE: {mae_lstm:.2f}")
print(f"⚡ Training Time: {train_time_lstm:.4f} sec | Prediction Time: {pred_time_lstm:.4f} sec\n")


In [ ]:
# Plot training & validation loss values
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='red')
plt.title('LSTM Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Model Comparsion

In [ ]:
plt.figure(figsize=(20, 10))

# XGBoost
plt.subplot(2, 2, 1)
plt.plot(y_test_xgb.index, y_test_xgb.values, label="Actual", linewidth=2)
plt.plot(y_test_xgb.index, y_pred_xgb, label="XGBoost", linestyle='--')
plt.title("XGBoost: Actual vs Predicted")
plt.xlabel("Date Index")
plt.ylabel("Sales (venda)")
plt.legend()
plt.grid(True)

# Random Forest
plt.subplot(2, 2, 2)
plt.plot(y_test_xgb.index, y_test_xgb.values, label="Actual", linewidth=2)
plt.plot(y_test_xgb.index, y_pred_rf, label="Random Forest", linestyle='--')
plt.title("Random Forest: Actual vs Predicted")
plt.xlabel("Date Index")
plt.ylabel("Sales (venda)")
plt.legend()
plt.grid(True)

# Linear Regression
plt.subplot(2, 2, 3)
plt.plot(y_test_xgb.index, y_test_xgb.values, label="Actual", linewidth=2)
plt.plot(y_test_xgb.index, y_pred_lr, label="Linear Regression", linestyle='--')
plt.title("Linear Regression: Actual vs Predicted")
plt.xlabel("Date Index")
plt.ylabel("Sales (venda)")
plt.legend()
plt.grid(True)

# LSTM
plt.subplot(2, 2, 4)
plt.plot(y_test_xgb.index[-len(y_pred_lstm):], y_test_lstm, label="Actual", linewidth=2)
plt.plot(y_test_xgb.index[-len(y_pred_lstm):], y_pred_lstm, label="LSTM", linestyle='--')
plt.title("LSTM: Actual vs Predicted")
plt.xlabel("Date Index")
plt.ylabel("Sales (venda)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()



In [ ]:
metrics_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost', 'LSTM'],
    'R2': [r2_lr, r2_rf, r2_xgb, r2_lstm],
    'MSE': [mse_lr, mse_rf, mse_xgb, mse_lstm],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb, rmse_lstm],
    'MAE': [mae_lr, mae_rf, mae_xgb, mae_lstm],
    'Train Time (s)': [train_time_lr, train_time_rf, train_time_xgb, train_time_lstm],
    'Pred Time (s)': [pred_time_lr, pred_time_rf, pred_time_xgb, pred_time_lstm]
})

print(metrics_df.round(4))


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 10))

axs[0, 0].bar(metrics_df['Model'], metrics_df['R2'], color='skyblue')
axs[0, 0].set_title('R² Score Comparison')
axs[0, 0].set_ylim(0, 1)
axs[0, 0].set_ylabel('R²')

axs[0, 1].bar(metrics_df['Model'], metrics_df['MSE'], color='salmon')
axs[0, 1].set_title('Mean Squared Error (MSE)')
axs[0, 1].set_ylabel('MSE')

axs[1, 0].bar(metrics_df['Model'], metrics_df['RMSE'], color='lightgreen')
axs[1, 0].set_title('Root Mean Squared Error (RMSE)')
axs[1, 0].set_ylabel('RMSE')

axs[1, 1].bar(metrics_df['Model'], metrics_df['MAE'], color='orange')
axs[1, 1].set_title('Mean Absolute Error (MAE)')
axs[1, 1].set_ylabel('MAE')

for ax in axs.flat:
    ax.grid(True, linestyle='--', alpha=0.7)

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# Efficiency Comparison
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].bar(metrics_df['Model'], metrics_df['Train Time (s)'], color='purple')
axs[0].set_title('Training Time Comparison')
axs[0].set_ylabel('Seconds')
axs[0].set_yscale('log')  # 🔑 log-scale for better visualization
axs[0].grid(True, linestyle='--', alpha=0.7)

axs[1].bar(metrics_df['Model'], metrics_df['Pred Time (s)'], color='teal')
axs[1].set_title('Prediction Time Comparison')
axs[1].set_ylabel('Seconds')
axs[1].set_yscale('log')  # 🔑 log-scale
axs[1].grid(True, linestyle='--', alpha=0.7)

plt.suptitle('Computational Efficiency Comparison', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()
